In [24]:
import torch
from torchvision import datasets

In [25]:
mnist_train = datasets.MNIST(root="data", train=True, download=True)
mnist_test = datasets.MNIST(root="data", train=False, download=True)

In [26]:
X_train = ((mnist_train.data)/255.).reshape(60000, 784)
y_train = mnist_train.targets
X_test = ((mnist_test.data)/255.).reshape(10000, 784)
y_test = mnist_test.targets

In [27]:
W = torch.zeros(10, 784)
for i in range(10):
    W[i] = X_train[y_train==i].mean(dim=0)
    W[i] = W[i]/W[i].norm()

In [28]:
def error(y_hat, y):
    return (y_hat.argmax(dim=1) != y).float().mean()

def cross_entropy_loss(y_hat, y):
    loss = 0.0
    for i in range(len(y)):
        loss += -y_hat[i, y[i]] + y_hat[i].exp().sum().log()
    return loss / len(y)

def fast_cross_entropy_loss(y_hat, y):
    loss = -y_hat[torch.arange(len(y)), y] + torch.logsumexp(y_hat, dim=1)
    return loss.mean()

In [29]:
fast_cross_entropy_loss(X_test @ W.T, y_test)

tensor(1.1673)

In [30]:
W = torch.zeros(10, 784, requires_grad=True)
for t in range(20):
    fast_cross_entropy_loss(X_train @ W.T, y_train).backward()
    with torch.no_grad():
        W -= 0.5 * W.grad
    W.grad.zero_()
    

In [31]:
error(X_test @ W.T, y_test)

tensor(0.1297)

In [34]:
W = torch.zeros(10, 784, requires_grad=True)
for t in range(1):
    for i in range(60000):
        fast_cross_entropy_loss(X_train[i:i+1] @ W.T, y_train[i:i+1]).backward()
        with torch.no_grad():
            W -= 0.01 * W.grad
        W.grad.zero_()

In [35]:
error(X_test @ W.T, y_test)

tensor(0.0967)